# React 框架 · 第 06 天

## 极客头条后台管理项目实战

本篇依据已下载的 React 基础 day06 MD 笔记、PPT 和随堂代码整理。项目以 React + React Router + Redux Toolkit + Ant Design 为基础，完成从环境搭建、登录鉴权、后台布局，到文章发布、封面上传、文章列表、编辑删除和项目打包优化的完整流程。

## 一、学习目标

学完本篇后，应能：

- 搭建一个 React 后台管理项目并组织清晰的目录结构。
- 使用 Sass、Ant Design、CRACO 和路径别名提高开发效率。
- 封装 Axios 请求工具，统一处理 baseURL、响应数据和 token。
- 使用 Redux Toolkit 管理用户 token 和个人信息。
- 完成登录表单校验、登录跳转、token 持久化和路由鉴权。
- 使用 Ant Design Layout、Menu 和嵌套路由搭建后台框架。
- 完成文章发布、频道获取、富文本、封面上传和编辑回填。
- 完成文章列表筛选、分页、删除、编辑跳转和首页图表。
- 使用懒加载、体积分析和 CDN 优化生产构建。

## 二、项目环境和目录设计

### 创建项目

```bash
npx create-react-app react-jike
cd react-jike
npm start
```

推荐的 `src` 目录：

```text
src/
├─ apis/          # 接口函数
├─ assets/        # 图片、字体等静态资源
├─ components/    # 通用组件，如 AuthRoute
├─ pages/         # 页面组件
│  ├─ Home/
│  ├─ Layout/
│  ├─ Login/
│  ├─ Article/
│  └─ Publish/
├─ store/         # Redux 状态管理
├─ utils/         # request、token 等工具
├─ App.js
├─ index.js
└─ index.scss
```

入口负责挂载根组件，页面和业务逻辑放到对应目录，通用能力不要散落在页面文件中。项目一开始就规划目录，可以减少后期“所有代码都堆在 App.js”的问题。

## 三、Sass、Ant Design 与基础样式

### Sass

```bash
npm i -D sass
```

创建 `index.scss` 后可以使用嵌套写法：

```scss
body {
  margin: 0;

  .app-title {
    color: #1890ff;
  }
}
```

### Ant Design

```bash
npm i antd @ant-design/icons
```

```jsx
import { Button } from 'antd'

function Login() {
  return <Button type="primary">测试按钮</Button>
}
```

### Reset

```bash
npm i normalize.css
```

```css
html,
body,
#root {
  height: 100%;
  margin: 0;
}
```

Ant Design 负责通用后台组件，Sass 负责页面专属布局和样式，reset 负责消除浏览器默认差异。

## 四、配置 `@/` 路径别名

项目层级较深时，使用 `@/` 代替多层相对路径：

```bash
npm i -D @craco/craco
```

```js
// craco.config.js
const path = require('path')

module.exports = {
  webpack: {
    alias: {
      '@': path.resolve(__dirname, 'src'),
    },
  },
}
```

修改 `package.json`：

```json
{
  "scripts": {
    "start": "craco start",
    "build": "craco build",
    "test": "craco test"
  }
}
```

根目录增加 `jsconfig.json`，让编辑器也能联想：

```json
{
  "compilerOptions": {
    "baseUrl": "./",
    "paths": { "@/*": ["src/*"] }
  }
}
```

## 五、路由基础设计

项目使用 React Router 的嵌套路由：登录页独立，后台 Layout 作为受保护的父路由，Home、Article、Publish 作为子路由。

```jsx
const router = createBrowserRouter([
  {
    path: '/',
    element: (
      <AuthRoute>
        <Layout />
      </AuthRoute>
    ),
    children: [
      { index: true, element: <Home /> },
      { path: 'article', element: <Article /> },
      { path: 'publish', element: <Publish /> },
    ],
  },
  { path: '/login', element: <Login /> },
])
```

Layout 中必须渲染 `<Outlet />`，子路由页面才会出现在内容区域。登录页不需要后台侧边栏，因此作为独立一级路由。

## 六、封装 Axios request 工具

统一的请求工具可以集中配置接口前缀、超时时间、请求拦截器和响应处理：

```js
// utils/request.js
import axios from 'axios'
import { getToken, clearToken } from './token'

const http = axios.create({
  baseURL: 'http://geek.itheima.net/v1_0',
  timeout: 5000,
})

http.interceptors.request.use(config => {
  const token = getToken()
  if (token) {
    config.headers.Authorization = `Bearer ${token}`
  }
  return config
})

http.interceptors.response.use(
  response => response.data,
  error => {
    if (error.response?.status === 401) {
      clearToken()
    }
    return Promise.reject(error)
  }
)

export { http }
```

统一返回 `response.data` 后，业务代码可以直接使用接口数据。拦截器只处理通用逻辑，页面级错误提示和业务判断仍放在调用方或统一错误处理层。

## 七、登录页面结构

登录页使用 Card、Form、Input 和 Button：

```jsx
import { Card, Form, Input, Button } from 'antd'
import logo from '@/assets/logo.png'
import './index.scss'

function Login() {
  return (
    <div className="login">
      <Card className="login-container">
        <img className="login-logo" src={logo} alt="极客头条" />
        <Form>
          <Form.Item name="mobile">
            <Input size="large" placeholder="请输入手机号" />
          </Form.Item>
          <Form.Item name="code">
            <Input size="large" placeholder="请输入验证码" />
          </Form.Item>
          <Form.Item>
            <Button type="primary" htmlType="submit" block size="large">
              登录
            </Button>
          </Form.Item>
        </Form>
      </Card>
    </div>
  )
}
```

页面背景、居中卡片、logo 和表单间距通过 `Login/index.scss` 实现。先完成静态结构，再接入校验和接口，调试范围更容易控制。

## 八、Ant Design 表单校验和取值

为 `Form.Item` 设置 `name` 和 `rules`，表单才能收集并校验字段：

```jsx
<Form
  validateTrigger={['onBlur']}
  onFinish={onFinish}
  initialValues={{ mobile: '', code: '' }}
>
  <Form.Item
    name="mobile"
    rules={[
      { required: true, message: '请输入手机号' },
      { pattern: /^1[3-9]\d{9}$/, message: '手机号码格式不对' },
    ]}
  >
    <Input size="large" placeholder="请输入手机号" />
  </Form.Item>
  <Form.Item
    name="code"
    rules={[{ required: true, message: '请输入验证码' }]}
  >
    <Input maxLength={6} size="large" placeholder="请输入验证码" />
  </Form.Item>
  <Button type="primary" htmlType="submit">登录</Button>
</Form>

const onFinish = values => {
  console.log(values)
}
```

`onFinish` 只会在校验通过后触发，参数 `values` 是表单字段对象。`validateTrigger` 可以配置为 `onBlur`、`onChange` 或它们的组合。

## 九、Redux 管理用户 token

用户 token 需要被请求拦截器、路由鉴权和页面共同使用，放入 Redux，并同时持久化到 localStorage：

```js
// utils/token.js
const TOKEN_KEY = 'geek_token'
export const setToken = token => localStorage.setItem(TOKEN_KEY, token)
export const getToken = () => localStorage.getItem(TOKEN_KEY)
export const clearToken = () => localStorage.removeItem(TOKEN_KEY)
```

```js
// store/modules/user.js
const userStore = createSlice({
  name: 'user',
  initialState: {
    token: getToken() || '',
    userInfo: {},
  },
  reducers: {
    setUserToken(state, action) {
      state.token = action.payload
      setToken(state.token)
    },
    setUserInfo(state, action) {
      state.userInfo = action.payload
    },
    clearUserInfo(state) {
      state.token = ''
      state.userInfo = {}
      clearToken()
    },
  },
})
```

Redux 负责内存中的即时读取，localStorage 负责刷新后的恢复；两者要保持同步。

## 十、登录流程、用户信息与退出

### 登录请求

```js
export const fetchLogin = loginForm => async dispatch => {
  const response = await http.post('/authorizations', loginForm)
  dispatch(setUserToken(response.data.token))
}
```

```jsx
const onFinish = async formValue => {
  await dispatch(fetchLogin(formValue))
  navigate('/')
  message.success('登录成功')
}
```

### 获取用户信息

```js
export const fetchUserInfo = () => async dispatch => {
  const response = await http.get('/user/profile')
  dispatch(setUserInfo(response.data))
}
```

Layout 加载后 dispatch，Header 通过 `useSelector` 展示用户名。

### 退出登录

```jsx
const loginOut = () => {
  dispatch(clearUserInfo())
  navigate('/login', { replace: true })
}

<Popconfirm
  title="是否确认退出？"
  okText="退出"
  cancelText="取消"
  onConfirm={loginOut}
>
  <LogoutOutlined /> 退出
</Popconfirm>
```

## 十一、路由鉴权和 token 失效

封装 `AuthRoute`，有 token 才渲染受保护页面，否则重定向到登录页：

```jsx
import { Navigate } from 'react-router-dom'
import { getToken } from '@/utils/token'

function AuthRoute({ children }) {
  return getToken() ? children : <Navigate to="/login" replace />
}
```

路由中包裹 Layout：

```jsx
{
  path: '/',
  element: (
    <AuthRoute>
      <Layout />
    </AuthRoute>
  ),
}
```

如果接口返回 401，说明 token 失效。响应拦截器应清除本地 token，并导航到登录页：

```js
if (error.response?.status === 401) {
  clearToken()
  router.navigate('/login')
}
```

鉴权组件解决“进入页面前”的拦截，响应拦截器解决“请求过程中失效”的处理，两者缺一不可。

## 十二、后台 Layout 与菜单路由

使用 Ant Design 的 `Layout`、`Header`、`Sider`、`Menu` 和 `Popconfirm` 搭建后台框架：

```jsx
const items = [
  { label: '首页', key: '/', icon: <HomeOutlined /> },
  { label: '文章管理', key: '/article', icon: <DiffOutlined /> },
  { label: '创建文章', key: '/publish', icon: <EditOutlined /> },
]

function GeekLayout() {
  const navigate = useNavigate()
  const location = useLocation()

  return (
    <Layout>
      <Header>头条后台</Header>
      <Layout>
        <Sider>
          <Menu
            mode="inline"
            items={items}
            selectedKeys={[location.pathname]}
            onClick={item => navigate(item.key)}
          />
        </Sider>
        <Layout className="layout-content">
          <Outlet />
        </Layout>
      </Layout>
    </Layout>
  )
}
```

`selectedKeys` 使用当前 pathname，可让刷新、前进、后退和手动输入地址后菜单仍然正确高亮。`Outlet` 是二级页面的渲染出口。

## 十三、首页数据图表：ECharts

首页适合用图表展示数据概览。先安装 ECharts：

```bash
npm i echarts
```

基础图表组件：

```jsx
import { useEffect, useRef } from 'react'
import * as echarts from 'echarts'

function BarChart({ xData, sData, style = { width: 400, height: 300 } }) {
  const chartRef = useRef(null)

  useEffect(() => {
    const chart = echarts.init(chartRef.current)
    chart.setOption({
      xAxis: { type: 'category', data: xData },
      yAxis: { type: 'value' },
      series: [{ data: sData, type: 'bar' }],
    })

    return () => chart.dispose()
  }, [xData, sData])

  return <div ref={chartRef} style={style} />
}
```

使用：

```jsx
<BarChart xData={['Vue', 'React', 'Angular']} sData={[2000, 5000, 1000]} />
```

`useRef` 保存 DOM 容器，`useEffect` 在渲染后初始化图表，依赖变化时更新，卸载时 `dispose`，避免实例和事件泄漏。

## 十四、文章发布：基础表单和富文本

发布页使用 Card、Breadcrumb、Form、Select、Radio、Upload 和富文本编辑器：

```bash
npm i react-quill@2.0.0-beta.2
```

```jsx
import ReactQuill from 'react-quill'
import 'react-quill/dist/quill.snow.css'

<Form
  labelCol={{ span: 4 }}
  wrapperCol={{ span: 16 }}
  onFinish={onFinish}
>
  <Form.Item
    label="标题"
    name="title"
    rules={[{ required: true, message: '请输入文章标题' }]}
  >
    <Input placeholder="请输入文章标题" />
  </Form.Item>
  <Form.Item label="频道" name="channel_id" rules={[{ required: true }]}>
    <Select placeholder="请选择文章频道">{channelOptions}</Select>
  </Form.Item>
  <Form.Item label="内容" name="content" rules={[{ required: true }]}>
    <ReactQuill className="publish-quill" theme="snow" />
  </Form.Item>
  <Button type="primary" htmlType="submit">发布文章</Button>
</Form>
```

频道列表通过 `http.get('/channels')` 请求后渲染为 Select 选项。富文本是受控表单字段，提交时和标题、频道一起生成文章请求体。

## 十五、文章封面上传与受控列表

封面类型有无图、单图和三图，使用 `Radio.Group` 选择，使用 `Upload` 上传：

```jsx
const [imageType, setImageType] = useState(0)
const [imageList, setImageList] = useState([])
const cacheImageList = useRef([])

const onUploadChange = info => {
  setImageList(info.fileList)
  cacheImageList.current = info.fileList
}

const onTypeChange = event => {
  const type = event.target.value
  setImageType(type)
  if (type === 0) setImageList([])
  if (type === 1) {
    setImageList(cacheImageList.current[0] ? [cacheImageList.current[0]] : [])
  }
  if (type === 3) setImageList(cacheImageList.current)
}
```

```jsx
<Radio.Group value={imageType} onChange={onTypeChange}>
  <Radio value={1}>单图</Radio>
  <Radio value={3}>三图</Radio>
  <Radio value={0}>无图</Radio>
</Radio.Group>

{imageType > 0 && (
  <Upload
    name="image"
    action="http://geek.itheima.net/v1_0/upload"
    listType="picture-card"
    fileList={imageList}
    onChange={onUploadChange}
    maxCount={imageType}
    multiple={imageType > 1}
  >
    <PlusOutlined />
  </Upload>
)}
```

`fileList` 传入后 Upload 成为受控组件；`useRef` 保存完整上传记录，切换单图/三图时可以恢复之前上传的图片。

## 十六、发布文章请求体

提交前校验封面类型和图片数量，再把 Upload 返回的 URL 转换为接口格式：

```jsx
const onFinish = async formValue => {
  if (imageType !== imageList.length) {
    return message.warning('图片类型和数量不一致')
  }

  const { channel_id, content, title } = formValue
  const params = {
    channel_id,
    content,
    title,
    type: imageType,
    cover: {
      type: imageType,
      images: imageList.map(file => file.response.data.url),
    },
  }

  await http.post('/mp/articles?draft=false', params)
  message.success('发布文章成功')
}
```

接口通常只需要图片 URL，不需要整个 Upload 文件对象。发布后可以清空表单、返回文章列表或跳转到详情页。生产代码还应处理上传失败、重复提交和请求错误。

## 十七、文章列表：筛选、频道和表格

文章列表页面分为筛选区和表格区。筛选表单可以包含状态、频道和日期范围：

```jsx
<Form initialValues={{ status: '' }} onFinish={onFinish}>
  <Form.Item label="状态" name="status">
    <Radio.Group>
      <Radio value="">全部</Radio>
      <Radio value={0}>草稿</Radio>
      <Radio value={2}>审核通过</Radio>
    </Radio.Group>
  </Form.Item>
  <Form.Item label="频道" name="channel_id">
    <Select placeholder="请选择文章频道">{channels}</Select>
  </Form.Item>
  <Form.Item label="日期" name="date">
    <RangePicker locale={locale} />
  </Form.Item>
  <Button type="primary" htmlType="submit">筛选</Button>
</Form>
```

表格列通过 `columns` 描述，封面、状态、时间和操作使用 `render` 自定义：

```jsx
const columns = [
  {
    title: '封面',
    dataIndex: 'cover',
    render: cover => <img src={cover.images[0] || img404} width={80} alt="" />,
  },
  { title: '标题', dataIndex: 'title' },
  { title: '发布时间', dataIndex: 'pubdate' },
  { title: '阅读数', dataIndex: 'read_count' },
]

<Table rowKey="id" columns={columns} dataSource={list} />
```

## 十八、文章列表数据、筛选与分页

列表请求参数和列表结果分别管理：

```jsx
const [article, setArticle] = useState({ list: [], count: 0 })
const [params, setParams] = useState({
  page: 1,
  per_page: 10,
  begin_pubdate: null,
  end_pubdate: null,
  status: null,
  channel_id: null,
})

useEffect(() => {
  async function fetchArticleList() {
    const response = await http.get('/mp/articles', { params })
    setArticle({
      list: response.data.results,
      count: response.data.total_count,
    })
  }
  fetchArticleList()
}, [params])
```

筛选时把表单值转换成接口字段：

```jsx
const onFinish = values => {
  const { channel_id, date, status } = values
  setParams(prev => ({
    ...prev,
    page: 1,
    status,
    channel_id,
    begin_pubdate: date?.[0]?.format('YYYY-MM-DD'),
    end_pubdate: date?.[1]?.format('YYYY-MM-DD'),
  }))
}
```

分页时只修改 page，`params` 变化会触发重新请求：

```jsx
<Table
  pagination={{
    current: params.page,
    pageSize: params.per_page,
    total: article.count,
    onChange: page => setParams(prev => ({ ...prev, page })),
  }}
/>
```

## 十九、删除和编辑文章

删除是破坏性操作，应使用 `Popconfirm` 二次确认：

```jsx
const deleteArticle = async record => {
  await http.delete(`/mp/articles/${record.id}`)
  setParams(prev => ({ ...prev, page: 1 }))
  message.success('删除成功')
}

<Popconfirm
  title="确认删除该条文章吗？"
  okText="确认"
  cancelText="取消"
  onConfirm={() => deleteArticle(record)}
>
  <Button danger shape="circle" icon={<DeleteOutlined />} />
</Popconfirm>
```

编辑按钮把文章 id 放到查询参数中：

```jsx
const navigate = useNavigate()
<Button
  icon={<EditOutlined />}
  onClick={() => navigate(`/publish?id=${record.id}`)}
/>
```

发布页根据是否存在 `id`，决定当前是新增还是编辑。删除后重新请求列表，避免只修改前端显示而没有同步服务器。

## 二十、编辑文章：数据回填与更新

通过 `useSearchParams` 读取文章 id，再请求详情并回填表单：

```jsx
const [searchParams] = useSearchParams()
const articleId = searchParams.get('id')
const [form] = Form.useForm()

useEffect(() => {
  if (!articleId) return

  async function getArticle() {
    const response = await http.get(`/mp/articles/${articleId}`)
    const { cover, ...formValue } = response.data
    form.setFieldsValue({ ...formValue, type: cover.type })
    setImageType(cover.type)
    setImageList(cover.images.map(url => ({ url })))
  }

  getArticle()
}, [articleId, form])
```

提交时兼容新上传文件和旧的回填图片：

```jsx
const formatUrl = list => list.map(file =>
  file.response ? file.response.data.url : file.url
)

const data = {
  channel_id,
  content,
  title,
  type: imageType,
  cover: { type: imageType, images: formatUrl(imageList) },
}

if (articleId) {
  await http.put(`/mp/articles/${articleId}?draft=false`, data)
} else {
  await http.post('/mp/articles?draft=false', data)
}
```

页面标题、按钮文案和提交提示可以根据 `articleId` 在“发布文章”和“编辑文章”之间切换。

## 二十一、项目打包与本地预览

生产构建：

```bash
npm run build
```

构建结果通常输出到 `build/`。本地预览可以安装 `serve`：

```bash
npm i -g serve
serve -s ./build
```

然后访问 `http://localhost:3000/`。预览时重点检查：

- 路由切换和刷新是否正常。
- 静态资源路径是否正确。
- 接口地址是否需要替换为生产环境。
- history 路由是否配置服务器回退。
- 上传、登录和鉴权流程是否仍然可用。

## 二十二、性能优化：路由懒加载

使用 `lazy` 和 `Suspense` 拆分页面代码，用户访问某个路由时再加载对应资源：

```jsx
import { lazy, Suspense } from 'react'

const Home = lazy(() => import('@/pages/Home'))
const Article = lazy(() => import('@/pages/Article'))
const Publish = lazy(() => import('@/pages/Publish'))

const lazyElement = Component => (
  <Suspense fallback={<div>加载中...</div>}>
    <Component />
  </Suspense>
)

const routes = [
  { path: '/', element: lazyElement(Home) },
  { path: '/article', element: lazyElement(Article) },
  { path: '/publish', element: lazyElement(Publish) },
]
```

登录页和 Layout 等首屏必需模块可以保留同步加载，文章列表、发布页等页面按路由拆分，减小首屏 JavaScript。

## 二十三、构建体积分析与 CDN

安装分析工具：

```bash
npm i -D source-map-explorer
```

在 `package.json` 中增加：

```json
{
  "scripts": {
    "analyze": "source-map-explorer 'build/static/js/*.js'"
  }
}
```

先构建，再运行 `npm run analyze`，通过可视化图表找出体积较大的依赖。

对于 React、ReactDOM 等稳定的大包，可以在生产环境使用 CDN，并通过 CRACO 配置 webpack externals：

```js
whenProd(() => {
  webpackConfig.externals = {
    react: 'React',
    'react-dom': 'ReactDOM',
  }
})
```

同时在 `public/index.html` 注入对应 CDN 脚本。CDN 地址、版本和可用性必须由部署环境统一管理，不能只删除依赖而忘记注入全局变量。

## 二十四、常见问题与排查顺序

1. **登录后刷新仍回到登录页**：token 是否写入 localStorage，Redux 初始值是否读取了 token。
2. **接口返回 401**：请求拦截器是否注入 `Authorization: Bearer token`，token 是否过期。
3. **受保护页面没有拦截**：路由是否真的使用 `AuthRoute` 包裹 Layout。
4. **二级路由空白**：Layout 是否渲染 `<Outlet />`，子路由 path 是否正确。
5. **菜单高亮错误**：`selectedKeys` 是否使用 `location.pathname`。
6. **文章发布失败**：检查标题、频道、内容和封面数量，确认图片 URL 来自上传响应。
7. **编辑图片消失**：回填时将 URL 转换为 Upload 所需的 `fileList`，提交时兼容 `file.url` 和 `file.response`。
8. **图表重复或报错**：ECharts 初始化后在清理函数中 `dispose`。
9. **生产环境刷新 404**：history 路由需要服务器回退配置。
10. **别名导入失败**：CRACO 脚本、webpack alias 和 jsconfig 三处配置要一致。

排查顺序建议是：先看控制台和 Network，再确认 token、接口参数、Redux 状态和路由匹配，最后处理页面样式。

## 二十五、本日总结

- 通过 CRA、Sass、Ant Design、CRACO 和路径别名搭建后台项目基础。
- Axios 统一封装请求，拦截器注入 token 并处理 401 失效。
- Redux Toolkit 管理登录 token、用户信息和异步接口状态。
- AuthRoute 保护后台路由，Layout 通过菜单和 Outlet 组织页面。
- 发布模块完成富文本、频道加载、封面上传、单图/三图切换和文章提交。
- 文章列表完成筛选、分页、表格展示、删除和编辑跳转。
- 编辑页通过查询参数回填表单和封面，并复用新增接口逻辑。
- ECharts、懒加载、体积分析和 CDN 让项目具备基础的可视化与生产优化能力。

这一天的重点不是记住每个组件 API，而是掌握一个后台项目从初始化到上线前优化的完整开发链路。